In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

In [ ]:
ticker = "AAPL"

df = yf.download(
    ticker,
    start="2015-01-01",
    end="2025-01-01"
)

df.head()

In [ ]:
df.info()

df.describe()

In [ ]:
plt.figure(figsize=(15,6))

plt.plot(df["Close"])

plt.title("Apple Closing Price")
plt.show()

In [ ]:
df["MA20"] = df["Close"].rolling(20).mean()

In [ ]:
df["MA50"] = df["Close"].rolling(50).mean()

In [ ]:
df["Return"] = df["Close"].pct_change()

In [ ]:
delta = df["Close"].diff()

gain = delta.where(delta > 0, 0)
loss = -delta.where(delta < 0, 0)

avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()

rs = avg_gain / avg_loss

df["RSI"] = 100 - (100 / (1 + rs))

In [ ]:
ema12 = df["Close"].ewm(span=12).mean()

ema26 = df["Close"].ewm(span=26).mean()

df["MACD"] = ema12 - ema26

In [ ]:
df = df.dropna()

df.head()

In [ ]:
df["Target"] = df["Close"].shift(-1)

df = df.dropna()

In [ ]:
features = [
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "MA20",
    "MA50",
    "RSI",
    "MACD"
]

X = df[features]

y = df["Target"]

In [ ]:
split = int(len(df) * 0.8)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

In [ ]:
scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [ ]:
lr = LinearRegression()

lr.fit(
    X_train_scaled,
    y_train
)

lr_pred = lr.predict(
    X_test_scaled
)

In [ ]:
lr_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        lr_pred
    )
)

lr_mae = mean_absolute_error(
    y_test,
    lr_pred
)

lr_r2 = r2_score(
    y_test,
    lr_pred
)

print("RMSE:", lr_rmse)
print("MAE:", lr_mae)
print("R2:", lr_r2)

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

rf.fit(
    X_train,
    y_train
)

rf_pred = rf.predict(
    X_test
)

In [ ]:
rf_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        rf_pred
    )
)

rf_mae = mean_absolute_error(
    y_test,
    rf_pred
)

rf_r2 = r2_score(
    y_test,
    rf_pred
)

print("RMSE:", rf_rmse)
print("MAE:", rf_mae)
print("R2:", rf_r2)

In [ ]:
close_data = df[["Close"]]

scaler_lstm = MinMaxScaler()

scaled_close = scaler_lstm.fit_transform(
    close_data
)

In [ ]:
X_lstm = []
y_lstm = []

for i in range(60, len(scaled_close)):
    X_lstm.append(
        scaled_close[i-60:i, 0]
    )

    y_lstm.append(
        scaled_close[i, 0]
    )

X_lstm = np.array(X_lstm)
y_lstm = np.array(y_lstm)

X_lstm = X_lstm.reshape(
    X_lstm.shape[0],
    X_lstm.shape[1],
    1
)

In [ ]:
split_lstm = int(
    len(X_lstm) * 0.8
)

X_train_lstm = X_lstm[:split_lstm]
X_test_lstm = X_lstm[split_lstm:]

y_train_lstm = y_lstm[:split_lstm]
y_test_lstm = y_lstm[split_lstm:]

In [ ]:
model = Sequential()

model.add(
    LSTM(
        50,
        return_sequences=True,
        input_shape=(60,1)
    )
)

model.add(
    Dropout(0.2)
)

model.add(
    LSTM(50)
)

model.add(
    Dropout(0.2)
)

model.add(Dense(25))

model.add(Dense(1))

model.compile(
    optimizer="adam",
    loss="mse"
)

In [ ]:
history = model.fit(
    X_train_lstm,
    y_train_lstm,
    epochs=20,
    batch_size=32,
    validation_data=(
        X_test_lstm,
        y_test_lstm
    )
)

In [ ]:
lstm_pred = model.predict(
    X_test_lstm
)

lstm_pred = scaler_lstm.inverse_transform(
    lstm_pred
)

actual = scaler_lstm.inverse_transform(
    y_test_lstm.reshape(-1,1)
)

lstm_rmse = np.sqrt(
    mean_squared_error(
        actual,
        lstm_pred
    )
)

lstm_mae = mean_absolute_error(
    actual,
    lstm_pred
)

lstm_r2 = r2_score(
    actual,
    lstm_pred
)

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest",
        "LSTM"
    ],
    "RMSE": [
        lr_rmse,
        rf_rmse,
        lstm_rmse
    ],
    "MAE": [
        lr_mae,
        rf_mae,
        lstm_mae
    ],
    "R2": [
        lr_r2,
        rf_r2,
        lstm_r2
    ]
})

results.sort_values(
    "RMSE"
)

In [ ]:
results.plot(
    x="Model",
    y="RMSE",
    kind="bar",
    figsize=(8,5)
)

plt.title("Model Comparison")
plt.show()

In [ ]:
best_model = results.loc[
    results["RMSE"].idxmin()
]

best_model